# Train the colony detector (RF-DETR) on a GPU

**Before running:** Runtime → Change runtime type → **T4 GPU** (or L4/A100 if you have them) → Save.
Then Runtime → **Run all**. Allow Google Drive access when asked (training is saved there so a
disconnect does not lose it).

Steps: download the labelled plates → build full-resolution tiles (test plates held out) →
fine-tune **RF-DETR Medium** → fit the confidence cut-off on validation plates → report accuracy
on the held-out test plates → save `rfdetr_tiles.pth` + `detector.json` to Drive.

In [ ]:
# 1. GPU check, Drive (for checkpoints), code
!nvidia-smi -L
from google.colab import drive
drive.mount('/content/drive')
OUT = '/content/drive/MyDrive/colony_detector_run'
import os; os.makedirs(OUT, exist_ok=True)
!rm -rf /content/colony_detector && git clone -q https://github.com/akashbabu9392/colony_detector.git /content/colony_detector
%cd /content/colony_detector
!pip -q install -r requirements.txt "rfdetr[train]>=1.7" pyyaml

In [ ]:
# 2. Dataset (~3.2 GB) from the GitHub release  (~2-4 min)
!mkdir -p /content/data && cd /content/data && \
  wget -q -nc https://github.com/akashbabu9392/colony_detector/releases/download/dataset-v1/data.set.zip && \
  wget -q -nc https://github.com/akashbabu9392/colony_detector/releases/download/dataset-v1/data.set.2.zip && \
  unzip -q -n data.set.zip -d raw && unzip -q -n data.set.2.zip -d raw && ls raw

In [ ]:
# 3. Full-resolution tiles + one whole-plate view per plate (test split held out)  (~15-25 min)
!python tools/build_training_set.py --dataset /content/data/raw --out /content/tiles --scale 1.0 \
    --empty-keep 0.15 --coco --gold "" | tail -1

In [ ]:
# 4. Train RF-DETR Medium (~2-4 h on a T4). If Colab disconnects: run cells 1-3 again,
#    then run this cell with RESUME = True to continue from the last saved epoch.
RESUME = False
resume = f"--resume {OUT}/rfdetr/last.ckpt" if RESUME else ""
!python tools/train_rfdetr.py --dataset /content/tiles/coco --size medium --resolution 640 \
    --epochs 15 --batch 4 --grad-accum 4 --lr 1e-4 --output {OUT}/rfdetr --out models/rfdetr_tiles.pth {resume}

In [ ]:
# 5. Fit the confidence cut-off on the VALIDATION plates, then measure on the TEST plates
os.environ['CD_RFDETR_WEIGHTS'] = 'models/rfdetr_tiles.pth'
!python tools/tune_detector.py --dataset /content/data/raw --engine rfdetr --split valid --conf-only --cache {OUT}/valid_dets.pkl | tail -4
!CD_ENGINES=rfdetr python tools/evaluate.py --dataset /content/data/raw --split test --out {OUT}/test_results.csv | tail -25

In [ ]:
# 6. Save the model to Drive (MyDrive/colony_detector_run) and download it
!cp models/rfdetr_tiles.pth models/detector.json {OUT}/
from google.colab import files
files.download('models/rfdetr_tiles.pth'); files.download('models/detector.json')